In [1]:
import librosa
import numpy as np
from spafe.features.lfcc import lfcc
from spafe.utils.preprocessing import SlidingWindow
from skimage.feature import local_binary_pattern


In [ ]:
def extract_lfcc(
    signal,
    sr=16000,
    num_ceps=20,
    nfilts=46,
    nfft=512,        # 25 ms @16k
    win_len=0.025,
    win_hop=0.01
):
    window = SlidingWindow(
        win_len=win_len,
        win_hop=win_hop,
        win_type="hamming"
    )
    lfcc_feat = lfcc(
        sig=signal,
        fs=sr,
        num_ceps=num_ceps,
        nfilts=nfilts,
        nfft=nfft,
        window=window
    )
    # shape: (frames, ceps)
    if lfcc_feat.shape[0] < 9:
        return None

    lfcc_T = lfcc_feat.T

    # delta
    lfcc_delta = librosa.feature.delta(lfcc_T)

    # delta-delta
    lfcc_delta2 = librosa.feature.delta(lfcc_T, order=2)

    # stack
    lfcc_all = np.vstack([lfcc_T, lfcc_delta, lfcc_delta2]).T
    
    return lfcc_all   # shape: (num_frames, num_ceps)


In [ ]:
import numpy as np
import librosa
from skimage.feature import local_binary_pattern


def build_lfcc_dataset(
    wav_list,
    sr=16000,
    segment_sec=2,
    out_path="X_lfcc.npy"
):
    all_feats = []

    segment_len = int(segment_sec * sr)

    for wav_path in wav_list:
        print("Processing:", wav_path)

        y, sr = librosa.load(wav_path, sr=sr)

        # nếu audio < 2s
        if len(y) < segment_len:
            segments = [y]
        else:
            num_segments = len(y) // segment_len
            segments = [
                y[i*segment_len:(i+1)*segment_len]
                for i in range(num_segments)
            ]

        for segment in segments:
            lfcc = extract_lfcc(segment, sr)
            if lfcc is None: continue
            lbp = local_binary_pattern(
                lfcc,
                P=8,
                R=1,
                method="default"
            )

            hist, _ = np.histogram(
                lbp.ravel(),
                bins=256,
                range=(0, 256)
            )

            all_feats.append(hist)

        print(f"{wav_path} -> {len(segments)} segments")

    X = np.vstack(all_feats)

    np.save(out_path, X)

    print("Saved:", out_path, X.shape)

    return X

In [8]:
import os
CLASS = "bonafide"
SET = "eval"

DIR = f"E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech"

wav_list = [
    os.path.join(DIR, f)
    for f in os.listdir(DIR)
    if f.endswith(".wav")
]
print(wav_list)
X_mfcc = build_lfcc_dataset(
    wav_list,
    out_path=f"E:/PythonFile/Project/Voice-Activity-Detect/data/feature/train/lfcc_lbp_speech_aurora.npy"
)

print(len(wav_list))
print(X_mfcc.shape)

['E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech\\1.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech\\10.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech\\100.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech\\101.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech\\102.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech\\103.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech\\104.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech\\105.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech\\106.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech\\107.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech\\108.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/